# 8.4 低秩分解 (Low-Rank Decomposition)

低秩分解将大权重矩阵分解为两个小矩阵的乘积，减少参数量和计算量。在LLM推理优化中，低秩分解可以显著降低模型大小和推理延迟。

本节涵盖：
- SVD分解与近似误差
- 低秩线性层实现
- 逐层最优rank选择
- 与LoRA/量化的组合策略

## 1. SVD分解与近似误差

**目的**：理解SVD分解的原理和rank-accuracy权衡

**基本原理**：
- 任意矩阵W∈R^(m×n)可分解为W = UΣV^T，其中U∈R^(m×m)，Σ∈R^(m×n)，V∈R^(n×n)
- 截断SVD保留前r个奇异值：W_r = U[:,:r] × diag(σ₁,...,σᵣ) × V^T[:r,:]
- Eckart-Young定理：W_r是rank-r的最优Frobenius范数近似

**参数量变化**：
- 原始：m×n参数
- 分解后：m×r + r×n = (m+n)×r参数
- 压缩比：mn / ((m+n)r)，当r << min(m,n)时压缩比大

**与LoRA的区别**：
- 低秩分解：**替换**原始权重，减少参数和推理计算
- LoRA：在原始权重旁**添加**低秩增量，不减少推理计算（合并后等效）

In [ ]:
import torch
import torch.nn as nn
import time

torch.manual_seed(42)

W = torch.randn(1024, 1024)
U, S, Vh = torch.linalg.svd(W, full_matrices=False)

print('=== SVD Decomposition & Approximation Error ===')
print(f'Original matrix: {W.shape}, params={W.numel():,}')
print(f'\nSingular value statistics:')
print(f'  Top-10: {S[:10].tolist()[:5]}...')
print(f'  Energy: top-64 captures {(S[:64]**2).sum()/(S**2).sum():.1%} of total')
print(f'  Energy: top-128 captures {(S[:128]**2).sum()/(S**2).sum():.1%} of total')
print(f'  Energy: top-256 captures {(S[:256]**2).sum()/(S**2).sum():.1%} of total')

print(f'\n{"Rank":>5} {"Error":>8} {"Energy":>8} {"Compression":>12} {"Params":>10}')
for rank in [16, 32, 64, 128, 256, 512]:
    W_approx = U[:, :rank] @ torch.diag(S[:rank]) @ Vh[:rank, :]
    error = (W - W_approx).norm() / W.norm()
    energy = (S[:rank]**2).sum() / (S**2).sum()
    orig_params = 1024 * 1024
    decomposed_params = 1024 * rank * 2
    compression = orig_params / decomposed_params
    print(f'{rank:>5} {error:>8.4f} {energy:>8.1%} {compression:>10.1f}x {decomposed_params:>10,}')

print(f'\nKey: Lower rank = more compression but more approximation error.')
print(f'Singular value energy distribution determines the optimal rank cutoff.')
print(f'Most LLM weight matrices have rapidly decaying singular values, enabling effective compression.')

## 2. 低秩线性层实现

**目的**：实现可用于实际模型替换的低秩线性层

**实现要点**：
- 支持从预训练权重初始化（SVD分解）
- 支持偏置项
- 推理时等价于原始线性层（但参数更少）
- 可选：分解后微调恢复精度

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

torch.manual_seed(42)

class LowRankLinear(nn.Module):
    def __init__(self, in_features, out_features, rank=64, bias=True):
        super().__init__()
        self.U = nn.Linear(in_features, rank, bias=False)
        self.V = nn.Linear(rank, out_features, bias=bias)

    def forward(self, x):
        return self.V(self.U(x))

    @classmethod
    def from_linear(cls, linear, rank):
        in_f, out_f = linear.weight.shape[1], linear.weight.shape[0]
        lowrank = cls(in_f, out_f, rank, bias=linear.bias is not None)
        U, S, Vh = torch.linalg.svd(linear.weight.data, full_matrices=False)
        sqrt_S = torch.sqrt(S[:rank])
        lowrank.U.weight.data = (Vh[:rank].T * sqrt_S.unsqueeze(0)).T
        lowrank.V.weight.data = U[:, :rank] * sqrt_S.unsqueeze(0)
        if linear.bias is not None:
            lowrank.V.bias.data = linear.bias.data.clone()
        return lowrank

full_linear = nn.Linear(1024, 1024)
lowrank_64 = LowRankLinear.from_linear(full_linear, rank=64)
lowrank_128 = LowRankLinear.from_linear(full_linear, rank=128)

x = torch.randn(32, 1024)

print('=== Low-Rank Linear Layer ===')
full_out = full_linear(x)
lr64_out = lowrank_64(x)
lr128_out = lowrank_128(x)

error_64 = (full_out - lr64_out).norm() / full_out.norm()
error_128 = (full_out - lr128_out).norm() / full_out.norm()

full_params = sum(p.numel() for p in full_linear.parameters())
lr64_params = sum(p.numel() for p in lowrank_64.parameters())
lr128_params = sum(p.numel() for p in lowrank_128.parameters())

print(f'Full linear:  {full_params:>10,} params')
print(f'LowRank r=64: {lr64_params:>10,} params ({lr64_params/full_params:.1%}), error={error_64:.4f}')
print(f'LowRank r=128:{lr128_params:>10,} params ({lr128_params/full_params:.1%}), error={error_128:.4f}')

n_runs = 500
start = time.time()
for _ in range(n_runs):
    _ = full_linear(x)
full_time = (time.time() - start) / n_runs * 1000

start = time.time()
for _ in range(n_runs):
    _ = lowrank_64(x)
lr64_time = (time.time() - start) / n_runs * 1000

start = time.time()
for _ in range(n_runs):
    _ = lowrank_128(x)
lr128_time = (time.time() - start) / n_runs * 1000

print(f'\nInference time (batch=32, dim=1024):')
print(f'  Full:    {full_time:.3f} ms')
print(f'  LR-64:   {lr64_time:.3f} ms ({full_time/lr64_time:.2f}x)')
print(f'  LR-128:  {lr128_time:.3f} ms ({full_time/lr128_time:.2f}x)')

print(f'\nKey: from_linear() uses SVD to initialize low-rank approximation.')
print(f'Lower rank = fewer params but more error. Speed depends on (m+n)r vs mn.')
print(f'For m=n=1024, rank=64: 2*1024*64=131K vs 1024*1024=1M params (8x compression).')

## 3. 逐层最优rank选择

**目的**：为不同层选择不同的rank，在精度和压缩之间取得最优平衡

**核心观察**：不同层的权重矩阵有不同的低秩特性：
- **Attention Q/K/V**：通常低秩性较好，可以用较小的rank
- **FFN上投影**：低秩性较差，需要较大的rank
- **嵌入层**：低秩性最差，通常不分解

**rank选择策略**：
- **固定压缩比**：所有层用相同压缩比，rank = min(m,n) / compression_ratio
- **等误差分配**：调整rank使各层近似误差相等
- **敏感度分析**：逐层评估分解对下游任务的影响，敏感层用大rank

In [ ]:
import torch
import torch.nn as nn
import math

torch.manual_seed(42)

class SimpleTransformer(nn.Module):
    def __init__(self, d_model=256, n_heads=4, n_layers=4, vocab_size=1000):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList()
        for _ in range(n_layers):
            self.layers.append(nn.ModuleDict({
                'q_proj': nn.Linear(d_model, d_model),
                'k_proj': nn.Linear(d_model, d_model),
                'v_proj': nn.Linear(d_model, d_model),
                'o_proj': nn.Linear(d_model, d_model),
                'up_proj': nn.Linear(d_model, d_model*4),
                'down_proj': nn.Linear(d_model*4, d_model),
            }))
        self.head = nn.Linear(d_model, vocab_size)

model = SimpleTransformer()

print('=== Per-Layer Rank Selection ===')
print(f'\nSingular value analysis per layer type:')

layer_errors = {}
for name, param in model.named_parameters():
    if 'weight' not in name or param.dim() < 2:
        continue
    W = param.data
    U, S, Vh = torch.linalg.svd(W, full_matrices=False)
    total_energy = (S**2).sum()
    for target_energy in [0.9, 0.95, 0.99]:
        cumsum = torch.cumsum(S**2, dim=0)
        rank_needed = (cumsum < total_energy * target_energy).sum().item() + 1
        rank_needed = min(rank_needed, min(W.shape))
        key = (name, target_energy)
        layer_errors[key] = rank_needed

layer_types = {}
for name, param in model.named_parameters():
    if 'weight' not in name or param.dim() < 2:
        continue
    layer_type = name.split('.')[-2] if '.' in name else name
    if layer_type not in layer_types:
        layer_types[layer_type] = []
    for target_energy in [0.95, 0.99]:
        rank = layer_errors.get((name, target_energy), 0)
        layer_types[layer_type].append(rank)

print(f'{"Layer Type":>12} {"Shape":>15} {"Rank@95%":>10} {"Rank@99%":>10}')
for lt in ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'up_proj', 'down_proj']:
    if lt in layer_types:
        for name, param in model.named_parameters():
            if lt in name and 'weight' in name and param.dim() >= 2:
                ranks = layer_types[lt]
                print(f'{lt:>12} {str(param.shape):>15} {ranks[0]:>10} {ranks[1]:>10}')
                break

print(f'\n--- Rank Selection Strategy ---')
print(f'1. Fixed compression: all layers same ratio (simple, suboptimal)')
print(f'2. Equal error: adjust rank so each layer has same relative error')
print(f'3. Sensitivity-based: use higher rank for sensitive layers (best, needs eval data)')
print(f'\nKey: Different layer types have different low-rank properties.')
print(f'Attention projections are often more compressible than FFN layers.')
print(f'Sensitivity analysis is the gold standard but requires downstream evaluation.')

## 4. 与LoRA/量化的组合策略

**目的**：将低秩分解与其他压缩技术组合，实现更大压缩比

**组合策略**：

1. **低秩分解 + 微调恢复**：先分解，再在目标任务上微调恢复精度
2. **低秩分解 + 量化**：先分解到低秩，再对U/V矩阵量化
3. **低秩分解 + LoRA**：分解后用LoRA微调，进一步适配任务

**工业实践**：
- 低秩分解常用于**推理优化**（减少参数和计算量）
- LoRA常用于**微调**（参数高效适配）
- 量化常用于**部署**（减少显存和延迟）
- 三者可以组合：量化低秩模型 + LoRA微调

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class QuantizedLowRankLinear(nn.Module):
    def __init__(self, in_features, out_features, rank=64, quant_bits=8):
        super().__init__()
        self.rank = rank
        self.quant_bits = quant_bits
        self.U_weight = nn.Parameter(torch.randn(rank, in_features))
        self.V_weight = nn.Parameter(torch.randn(out_features, rank))
        self.U_scale = nn.Parameter(torch.ones(rank), requires_grad=False)
        self.V_scale = nn.Parameter(torch.ones(out_features), requires_grad=False)

    def quantize_weights(self):
        with torch.no_grad():
            for name, weight, scale in [('U', self.U_weight, self.U_scale),
                                         ('V', self.V_weight, self.V_scale)]:
                w_max = weight.abs().amax(dim=-1, keepdim=True)
                scale.copy_(w_max.squeeze() / (2**(self.quant_bits-1) - 1))
                q = torch.clamp(torch.round(weight / w_max * (2**(self.quant_bits-1) - 1)),
                               -(2**(self.quant_bits-1)), 2**(self.quant_bits-1) - 1)
                weight.copy_(q * w_max / (2**(self.quant_bits-1) - 1))

    def forward(self, x):
        return F.linear(F.linear(x, self.U_weight), self.V_weight)

class LowRankWithLoRA(nn.Module):
    def __init__(self, in_features, out_features, rank=64, lora_rank=8, lora_alpha=16):
        super().__init__()
        self.U = nn.Linear(in_features, rank, bias=False)
        self.V = nn.Linear(rank, out_features, bias=False)
        self.lora_A = nn.Linear(in_features, lora_rank, bias=False)
        self.lora_B = nn.Linear(lora_rank, out_features, bias=False)
        self.lora_alpha = lora_alpha
        self.lora_rank = lora_rank
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        base_out = self.V(self.U(x))
        lora_out = self.lora_B(self.lora_A(x)) * (self.lora_alpha / self.lora_rank)
        return base_out + lora_out

d = 512
full = nn.Linear(d, d)
lr_only = LowRankLinear.from_linear(full, rank=64) if hasattr(LowRankLinear, 'from_linear') else None

U, S, Vh = torch.linalg.svd(full.weight.data, full_matrices=False)
rank = 64
sqrt_S = torch.sqrt(S[:rank])
U_w = (Vh[:rank].T * sqrt_S.unsqueeze(0)).T
V_w = U[:, :rank] * sqrt_S.unsqueeze(0)

qlr = QuantizedLowRankLinear(d, d, rank=64, quant_bits=8)
qlr.U_weight.data = U_w.clone()
qlr.V_weight.data = V_w.clone()
qlr.quantize_weights()

lrl = LowRankWithLoRA(d, d, rank=64, lora_rank=8)
lrl.U.weight.data = U_w.clone()
lrl.V.weight.data = V_w.clone()

x = torch.randn(16, d)

print('=== Combined Compression Strategies ===')
full_out = full(x)
qlr_out = qlr(x)
lrl_out = lrl(x)

full_params = sum(p.numel() for p in full.parameters())
qlr_params = sum(p.numel() for p in qlr.parameters())
lrl_params = sum(p.numel() for p in lrl.parameters())

qlr_error = (full_out - qlr_out).norm() / full_out.norm()
lrl_error = (full_out - lrl_out).norm() / full_out.norm()

print(f'{"Method":>25} {"Params":>10} {"Ratio":>8} {"Error":>8}')
print(f'{"Full Linear":>25} {full_params:>10,} {"1.0x":>8} {"0.0000":>8}')
print(f'{"Quantized LR (8bit)":>25} {qlr_params:>10,} {qlr_params/full_params:>7.1%} {qlr_error:>8.4f}')
print(f'{"LowRank + LoRA":>25} {lrl_params:>10,} {lrl_params/full_params:>7.1%} {lrl_error:>8.4f}')

print(f'\n--- Strategy Selection Guide ---')
strategies = [
    ('LowRank only', 'Inference optimization', 'Moderate compression, needs fine-tuning'),
    ('LowRank + Quant', 'Edge deployment', 'Max compression, some accuracy loss'),
    ('LowRank + LoRA', 'Fine-tuning + inference', 'Best accuracy, LoRA adds params'),
    ('LowRank + Quant + LoRA', 'Full pipeline', 'Max flexibility, complex pipeline'),
]
for name, use_case, tradeoff in strategies:
    print(f'  {name:25s}: {use_case:25s} | {tradeoff}')

print(f'\nKey: Low-rank decomposition reduces both params and compute (unlike LoRA).')
print(f'Combining with quantization gives maximum compression for edge deployment.')
print(f'LoRA on top of low-rank allows task-specific adaptation after compression.')